In [8]:
!pip install pandas scikit-learn -q
print('Done! Tools installed.')

Done! Tools installed.


In [9]:
import pandas as pd
import random

random.seed(42)

templates = {
    "Billing": [
        "I was charged twice for my subscription this month.",
        "My invoice shows an incorrect amount, please review.",
        "I need a refund for the duplicate payment.",
        "Why was my credit card charged before the renewal date?",
        "Please cancel my subscription and confirm the refund.",
        "The discount coupon was not applied to my invoice.",
    ],
    "Technical": [
        "The application crashes every time I try to export a report.",
        "I cannot log in, it keeps showing an error 500.",
        "The sync between mobile and desktop app is not working.",
        "File upload fails at 90 percent every single time.",
        "The dashboard is not loading any data since this morning.",
        "API requests are returning a timeout error.",
    ],
    "Account": [
        "I forgot my password and the reset link is not arriving.",
        "I need to change the email address linked to my account.",
        "Please help me transfer ownership of this account to my colleague.",
        "My account was suspended and I don't know why.",
        "I want to merge two accounts into one.",
        "Two-factor authentication is not sending me a code.",
    ],
    "General": [
        "Can you tell me more about your enterprise pricing plans?",
        "Do you offer a free trial for the premium plan?",
        "Is there a mobile app available for this product?",
        "What are your customer support hours?",
        "I would like a demo of the reporting features.",
        "Do you have documentation for the REST API?",
    ],
}

urgent_markers = [
    "This is extremely urgent, my whole team is blocked.",
    "URGENT: production is down right now.",
    "This is critical and needs immediate attention.",
    "We are losing business because of this issue.",
]

def priority_for(category, text):
    if any(w in text.lower() for w in ["urgent", "critical", "immediately", "production is down"]):
        return "High"
    if category == "Technical":
        return random.choices(["High", "Medium", "Low"], weights=[0.4, 0.4, 0.2])[0]
    if category == "Billing":
        return random.choices(["High", "Medium", "Low"], weights=[0.2, 0.5, 0.3])[0]
    if category == "Account":
        return random.choices(["High", "Medium", "Low"], weights=[0.3, 0.4, 0.3])[0]
    return random.choices(["Medium", "Low"], weights=[0.3, 0.7])[0]

rows = []
for category, examples in templates.items():
    for _ in range(150):
        text = random.choice(examples)
        if random.random() < 0.15:
            text = random.choice(urgent_markers) + " " + text
        priority = priority_for(category, text)
        rows.append({"text": text, "category": category, "priority": priority})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Created {len(df)} example tickets.")
df.head(10)

Created 600 example tickets.


,text,category,priority
0,The discount coupon was not applied to my invo...,Billing,High
1,This is critical and needs immediate attention...,Account,High
2,What are your customer support hours?,General,Low
3,Why was my credit card charged before the rene...,Billing,High
4,File upload fails at 90 percent every single t...,Technical,Medium
5,The dashboard is not loading any data since th...,Technical,Medium
6,"This is extremely urgent, my whole team is blo...",Billing,High
7,Can you tell me more about your enterprise pri...,General,Low
8,I was charged twice for my subscription this m...,Billing,Medium
9,I need to change the email address linked to m...,Account,Low


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_cat_train, y_cat_test, y_pri_train, y_pri_test = train_test_split(
    df["text"], df["category"], df["priority"],
    test_size=0.2, random_state=42, stratify=df["category"]
)

print(f"Training on {len(X_train)} tickets, testing on {len(X_test)} tickets.")

Training on 480 tickets, testing on 120 tickets.


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

category_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

category_pipeline.fit(X_train, y_cat_train)

cat_preds = category_pipeline.predict(X_test)
print("How well did it do predicting CATEGORY?\n")
print(classification_report(y_cat_test, cat_preds))

How well did it do predicting CATEGORY?

              precision    recall  f1-score   support

     Account       1.00      1.00      1.00        30
     Billing       1.00      1.00      1.00        30
     General       1.00      1.00      1.00        30
   Technical       1.00      1.00      1.00        30

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120



In [12]:
priority_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

priority_pipeline.fit(X_train, y_pri_train)

pri_preds = priority_pipeline.predict(X_test)
print("How well did it do predicting PRIORITY?\n")
print(classification_report(y_pri_test, pri_preds))

How well did it do predicting PRIORITY?

              precision    recall  f1-score   support

        High       0.59      0.51      0.55        39
         Low       0.69      0.65      0.67        31
      Medium       0.49      0.56      0.52        50

    accuracy                           0.57       120
   macro avg       0.59      0.57      0.58       120
weighted avg       0.57      0.57      0.57       120



In [13]:
my_ticket = "My app crashes every time I export a report, this is urgent"

predicted_category = category_pipeline.predict([my_ticket])[0]
predicted_priority = priority_pipeline.predict([my_ticket])[0]

print(f"Ticket: {my_ticket}")
print(f"Predicted category: {predicted_category}")
print(f"Predicted priority: {predicted_priority}")

Ticket: My app crashes every time I export a report, this is urgent
Predicted category: Technical
Predicted priority: High


In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML

text_box = widgets.Textarea(
    value="",
    placeholder="Type a customer email here, e.g. 'I was charged twice for my subscription this month'",
    layout=widgets.Layout(width="100%", height="80px")
)

button = widgets.Button(description="Classify this email", button_style="primary")
output = widgets.Output()

def on_click(b):
    output.clear_output()
    with output:
        text = text_box.value.strip()
        if not text:
            print("Please type something in the box first.")
            return
        category = category_pipeline.predict([text])[0]
        priority = priority_pipeline.predict([text])[0]
        confidence = max(category_pipeline.predict_proba([text])[0])

        routing = {
            "Billing": "Finance and Billing Team",
            "Technical": "Engineering Support Team",
            "Account": "Account Management Team",
            "General": "Sales / General Inquiries Team",
        }

        display(HTML(f"""
        <div style='border:2px solid #4A6FA5; border-radius:10px; padding:15px; margin-top:10px; font-family:Arial;'>
            <p><b>Predicted Category:</b> {category}</p>
            <p><b>Predicted Priority:</b> {priority}</p>
            <p><b>Confidence:</b> {confidence*100:.0f}%</p>
            <p><b>Routed To:</b> {routing.get(category, "General Queue")}</p>
        </div>
        """))

button.on_click(on_click)
display(text_box, button, output)

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder="Type a customer email here, e.g. '…

Button(button_style='primary', description='Classify this email', style=ButtonStyle())

Output()